# Tutorial 3: explicit Target/Baseline comparison

This notebook resolves the metadata-derived Sample Groups into an explicit Comparison Membership, runs reaction and gene tests, installs the immutable comparison Analysis Artifact, and plots its statistical distributions.


## Before you run it

The notebook is self-contained and repeats the setup from the previous tutorials. It keeps the same Th17 Compass dataset so that only the analysis layer changes.

In [ ]:
# ruff: noqa: E402, I001
import os
import sys
import tempfile
import warnings
from pathlib import Path

from bokeh.io import output_notebook

os.environ.setdefault(
    "MPLCONFIGDIR",
    str(Path(tempfile.gettempdir()) / "metabis-matplotlib"),
)
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", message="IProgress not found.*")


def find_metabis_repo_root_for_import(start: Path) -> Path:
    direct_candidates = [start, *start.parents]
    nested_candidates = [candidate / "metabis" for candidate in direct_candidates]
    for candidate in [*direct_candidates, *nested_candidates]:
        if (candidate / "src" / "metabis").exists():
            return candidate
    raise FileNotFoundError("Could not locate the Metabis repository root.")


repo_root_for_import = find_metabis_repo_root_for_import(Path.cwd().resolve())
src_path = repo_root_for_import / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from metabis import (
    AnalysisWorkspace,
    ExperimentAnalysis,
    ExpressionData,
    ExpressionPreprocessingConfig,
    GroupComparison,
    GroupComparisonSpec,
    MetabolicModel,
    ReactionData,
    ReactionPreprocessingConfig,
    SampleMetadata,
    plot_statistical_histograms,
    plot_statistical_scatterplots,
)
from metabis.analysis_workspace import RevisionKind
from metabis.version import __version__ as metabis_version

output_notebook()


In [ ]:
def find_metabis_repo_root(start: Path) -> Path:
    direct_candidates = [start, *start.parents]
    nested_candidates = [candidate / "metabis" for candidate in direct_candidates]
    for candidate in [*direct_candidates, *nested_candidates]:
        if (
            (candidate / "pyproject.toml").exists()
            and (candidate / "demo").exists()
            and (candidate / "src" / "metabis").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the Metabis repository root.")


repo_root = find_metabis_repo_root(Path.cwd().resolve())
data_dir = repo_root / "data"
required_example_files = (
    "Th17_Compass_reactions.tsv",
    "Th17_Compass_expression.tsv",
    "Th17_Compass_metadata.tsv",
)
missing = [name for name in required_example_files if not (data_dir / name).is_file()]
if missing:
    raise FileNotFoundError(
        "Missing example files: "
        + ", ".join(missing)
        + ". Run `python data/download_datasets.py`."
    )

reaction_data_path = data_dir / "Th17_Compass_reactions.tsv"
expression_data_path = data_dir / "Th17_Compass_expression.tsv"
metadata_path = data_dir / "Th17_Compass_metadata.tsv"
target_group_id = "Th17p"
baseline_group_id = "Th17n"

model = MetabolicModel(use_cache=True)
model.apply_alignment_corrections(model.get_alignment_file_path("RECON2"))

reaction_data, reaction_validation = ReactionData.create_with_validation(
    str(reaction_data_path),
    data_mode="dual",
    metabolic_model=model,
    reaction_name_format="rxnRecon3DID",
    csv_separator="\t",
    samples_in="columns",
    check_old_identifiers=True,
    preprocessing=ReactionPreprocessingConfig(compass_activity=True),
)
expression_data, expression_validation = ExpressionData.create_with_validation(
    str(expression_data_path),
    gene_list=model.get_genes(),
    metabolic_model=model,
    gene_format="geneSymbols",
    preprocessing=ExpressionPreprocessingConfig(normalize_total=True),
    data_format="csv",
    csv_separator="\t",
    samples_in="columns",
)
metadata = SampleMetadata.from_table(
    metadata_path,
    sample_id_column="sample",
)

dm = AnalysisWorkspace(
    metabolic_model=model,
    reaction_data=reaction_data,
    expression_data=expression_data,
)
dm.update_sample_metadata(metadata)

print(f"Metabis version: {metabis_version}")
print(f"Loaded {len(reaction_data.get_sample_ids())} aligned samples")
grouping_result = dm.load_groups_from_metadata(["cell_type"])
print(f"Groups available for comparison: {dm.get_group_ids()}")


## 1. Resolve Comparison Membership and run the tests

`GroupComparisonSpec` names the Target and Baseline by canonical Sample Group ID. Resolution freezes the exact samples and comparison-local labels before analysis starts.


In [ ]:
analysis_dependencies = frozenset(
    {
        RevisionKind.DATASET,
        RevisionKind.METADATA,
        RevisionKind.GROUPS,
        RevisionKind.PREPROCESSING,
    }
)
analysis_snapshot = dm.capture_analysis_snapshot(analysis_dependencies)

analysis = ExperimentAnalysis(dm)
dm.install_experiment_analysis(analysis_snapshot.provenance, analysis)
dm.refresh_experiment_analysis_presentation()
dm.set_analysis_group(target_group_id)

comparison_spec = GroupComparisonSpec.pairwise(
    (target_group_id,),
    (baseline_group_id,),
    target_label="Pathogenic Th17",
    baseline_label="Non-pathogenic Th17",
)
group_comparison = GroupComparison(
    spec=comparison_spec,
    test="wilcoxon",
    correction="fdr_bh",
)
group_comparison.run_test(dm, for_genes=False)
group_comparison.run_test(dm, for_genes=True)

membership = group_comparison.membership
assert membership is not None
print(
    f"Target: {membership.target_label} ({len(membership.target_samples)} samples)"
)
print(
    f"Baseline: {membership.baseline_label} "
    f"({len(membership.baseline_samples)} samples)"
)


## 2. Define significance and install the Analysis Artifact

Thresholds are part of the result configuration. The completed comparison becomes an immutable artifact and is installed transactionally through the Analysis Workspace.


In [ ]:
analysis_settings = {
    "pval_threshold": 0.10,
    "cohens_d_threshold": 0.0,
    "log2_fc_threshold": 0.0,
    "mean_difference_threshold": 0.0,
    "keep_reversed": True,
}

significant_reactions, reaction_threshold_info = (
    group_comparison.define_significant_elements(
        pval_threshold=analysis_settings["pval_threshold"],
        cohens_d_threshold=analysis_settings["cohens_d_threshold"],
        log2_fc_threshold=analysis_settings["log2_fc_threshold"],
        mean_difference_threshold=analysis_settings["mean_difference_threshold"],
        keep_reversed=analysis_settings["keep_reversed"],
        for_genes=False,
    )
)
significant_genes, gene_threshold_info = (
    group_comparison.define_significant_elements(
        pval_threshold=analysis_settings["pval_threshold"],
        cohens_d_threshold=analysis_settings["cohens_d_threshold"],
        log2_fc_threshold=analysis_settings["log2_fc_threshold"],
        mean_difference_threshold=analysis_settings["mean_difference_threshold"],
        keep_reversed=analysis_settings["keep_reversed"],
        for_genes=True,
    )
)

comparison_snapshot = dm.capture_analysis_snapshot(analysis_dependencies)
comparison_artifact = group_comparison.build_artifact(
    provenance=comparison_snapshot.provenance,
    source_fingerprints=dict(dm.source_fingerprints),
    keep_reversed=analysis_settings["keep_reversed"],
)
dm.install_group_comparison_artifact(
    comparison_snapshot.provenance,
    comparison_artifact,
)
dm.refresh_group_comparison_presentation()

reaction_results = comparison_artifact.get_test_results(for_genes=False)
gene_results = comparison_artifact.get_test_results(for_genes=True)

print(f"Current mode: {dm.get_mode()}")
print(f"Comparison fingerprint: {comparison_artifact.membership.fingerprint}")
print(f"Significant reactions: {len(significant_reactions)}")
print(f"Significant genes: {len(significant_genes)}")


## 3. Inspect the artifact tables

The reaction and gene tables are ordinary DataFrames projected from the installed artifact, so standard pandas sorting, filtering, and export apply.


In [ ]:
reaction_results.sort_values("adjusted_pval")[
    ["adjusted_pval", "cohens_d", "log2_fc", "mean_difference", "subsystems"]
].head(10)

In [ ]:
gene_results.sort_values("adjusted_pval")[
    ["adjusted_pval", "cohens_d", "log2_fc", "mean_difference", "subsystems"]
].head(10)

## 4. Histogram view of the comparison

`plot_statistical_histograms()` summarizes the global distribution of the chosen metrics and overlays the currently active significance thresholds.

In [ ]:
plot_statistical_histograms(
    dm,
    parameters_to_plot=["pval", "cohens_d", "log2_fc"],
    return_plot=False,
    show_thresholds=True,
    for_genes=False,
)

## 5. Scatter views of the same metrics

`plot_statistical_scatterplots()` is useful when you want to see how p-values, effect sizes, and fold changes relate to each other for the same set of reactions.

In [ ]:
plot_statistical_scatterplots(
    dm,
    parameters_to_plot=["pval", "cohens_d", "log2_fc"],
    return_plot=False,
    show_thresholds=True,
    for_genes=False,
)

## Takeaways

The Analysis Workspace now owns one authoritative comparison artifact with exact Target/Baseline membership, thresholds, provenance, and full-precision reaction and gene tables.

The next notebook uses that artifact for subsystem ranking and graph-artifact rendering.
